In [22]:


%pip install -q "transformers==5.22.1" "accelerate==1.15.0" "bitsandbytes==0.50.2" "sentence-transformers==5.7.0" pandas openpyxl scikit-learn tqdm



In [23]:

import torch
import transformers
import accelerate
import bitsandbytes
import sentence_transformers
import huggingface_hub

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

print("\nGPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU not available. Enable T4 GPU.")


PyTorch: 2.14.1+cu130
Transformers: 5.22.1
Accelerate: 1.15.0
BitsAndBytes: 0.50.2
Sentence Transformers: 5.7.0
Hugging Face Hub: 1.33.0

GPU Available: True
GPU Name: Tesla T4


In [24]:

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is required. In Colab, select "
        "Runtime > Change runtime type > T4 GPU."
    )

print("GPU available:", torch.cuda.get_device_name(0))


GPU available: Tesla T4


In [25]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

print("Qwen loaded successfully!")
print("Model device:", model.device)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen loaded successfully!
Model device: cuda:0


In [26]:

import pandas as pd
from pathlib import Path
from google.colab import files

# 1. Upload dataset files
print("Upload the three Excel files:")
uploaded = files.upload()

# 2. Automatically identify uploaded files
ambiguity_types = [
    "Anaphoric",
    "Coordination",
    "MissingCondition"
]

datasets = {}

for ambiguity_type in ambiguity_types:

    matching_files = [
        filename
        for filename in uploaded.keys()
        if Path(filename).name.lower().startswith(
            ambiguity_type.lower()
        )
        and filename.lower().endswith(".xlsx")
    ]

    if len(matching_files) != 1:
        raise ValueError(
            f"Expected exactly one file for "
            f"{ambiguity_type}, found: {matching_files}"
        )

    datasets[ambiguity_type] = matching_files[0]

print("\nDetected files:")
for ambiguity_type, filename in datasets.items():
    print(f"{ambiguity_type}: {filename}")

# 3. Convert manual labels to binary labels
label_map = {
    "UA": 0,
    "A": 1,
    "NPMC": 0,
    "PMC": 1
}

# 4. Load and clean datasets
all_data = []

for ambiguity_type, filename in datasets.items():

    df = pd.read_excel(filename)

    required_columns = [
        "Raw Requirements",
        "Manual Evaluation"
    ]

    if not all(
        column in df.columns
        for column in required_columns
    ):
        raise ValueError(
            f"Missing required columns in {filename}. "
            f"Available columns: {list(df.columns)}"
        )

    df = df[required_columns].copy()

    df["Requirement"] = (
        df["Raw Requirements"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    df["Label"] = (
        df["Manual Evaluation"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .map(label_map)
    )

    df = df[
        (df["Requirement"] != "") &
        (df["Label"].notna())
    ].copy()

    df["Label"] = df["Label"].astype(int)
    df["Ambiguity_Type"] = ambiguity_type
    df["Row_Index"] = df.index.astype(int)

    all_data.append(
        df[
            [
                "Requirement",
                "Label",
                "Ambiguity_Type",
                "Row_Index"
            ]
        ]
    )

# 5. Combine all datasets
data = pd.concat(
    all_data,
    ignore_index=True
)

# 6. Display dataset statistics
print("\nDataset loaded successfully!")
print("Total requirements:", len(data))

print("\nDistribution by ambiguity type and label:")
print(
    pd.crosstab(
        data["Ambiguity_Type"],
        data["Label"],
        margins=True
    )
)

# 7. Validate dataset
expected_counts = {
    "Anaphoric": (94, 65),
    "Coordination": (85, 74),
    "MissingCondition": (82, 5)
}

for ambiguity_type, (expected_0, expected_1) in expected_counts.items():

    subset = data[
        data["Ambiguity_Type"] == ambiguity_type
    ]

    actual_0 = (subset["Label"] == 0).sum()
    actual_1 = (subset["Label"] == 1).sum()

    assert (actual_0, actual_1) == (
        expected_0, expected_1
    ), f"Unexpected label distribution for {ambiguity_type}"

assert len(data) == 405, "Unexpected dataset size!"

print("\nDataset validation passed!")


Upload the three Excel files:


Saving MissingCondition.xlsx to MissingCondition (2).xlsx
Saving Anaphoric.xlsx to Anaphoric (2).xlsx
Saving Coordination.xlsx to Coordination (2).xlsx

Detected files:
Anaphoric: Anaphoric (2).xlsx
Coordination: Coordination (2).xlsx
MissingCondition: MissingCondition (2).xlsx

Dataset loaded successfully!
Total requirements: 405

Distribution by ambiguity type and label:
Label               0    1  All
Ambiguity_Type                 
Anaphoric          94   65  159
Coordination       85   74  159
MissingCondition   82    5   87
All               261  144  405

Dataset validation passed!


In [27]:

from sklearn.model_selection import train_test_split

# Reproducibility
RANDOM_STATE = 42

# Stratification by ambiguity type and label
stratify_labels = (
    data["Ambiguity_Type"]
    + "_"
    + data["Label"].astype(str)
)

# 20% Development / 80% Test
development_df, test_df = train_test_split(
    data,
    test_size=0.80,
    random_state=RANDOM_STATE,
    stratify=stratify_labels
)

# Reset indices
development_df = development_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Display results
print("Development Set:", len(development_df))
print("Test Set:", len(test_df))

print("\nDevelopment distribution:")
print(
    pd.crosstab(
        development_df["Ambiguity_Type"],
        development_df["Label"],
        margins=True
    )
)

print("\nTest distribution:")
print(
    pd.crosstab(
        test_df["Ambiguity_Type"],
        test_df["Label"],
        margins=True
    )
)

# Validation
assert len(development_df) == 81
assert len(test_df) == 324

print("\nData splitting completed successfully!")


Development Set: 81
Test Set: 324

Development distribution:
Label              0   1  All
Ambiguity_Type               
Anaphoric         19  13   32
Coordination      17  15   32
MissingCondition  16   1   17
All               52  29   81

Test distribution:
Label               0    1  All
Ambiguity_Type                 
Anaphoric          75   52  127
Coordination       68   59  127
MissingCondition   66    4   70
All               209  115  324

Data splitting completed successfully!


In [28]:

# Ambiguity definitions used in all prompting strategies

TYPE_DEFINITIONS = {
    "Anaphoric": (
        "Anaphoric ambiguity occurs when a pronoun "
        "or referring expression has an unclear "
        "antecedent or could refer to multiple entities."
    ),

    "Coordination": (
        "Coordination ambiguity occurs when words "
        "connected by conjunctions such as 'and' or 'or' "
        "allow multiple interpretations of grouping, "
        "scope, or relationships."
    ),

    "MissingCondition": (
        "Missing Condition ambiguity occurs when "
        "a requirement omits a condition, constraint, "
        "or triggering circumstance needed to "
        "determine when or how the specified "
        "behavior should occur."
    )
}

# Validate definitions
assert set(TYPE_DEFINITIONS.keys()) == set(
    data["Ambiguity_Type"].unique()
)

print("Ambiguity definitions loaded successfully!")

for ambiguity_type, definition in TYPE_DEFINITIONS.items():
    print(f"\n{ambiguity_type}:")
    print(definition)


Ambiguity definitions loaded successfully!

Anaphoric:
Anaphoric ambiguity occurs when a pronoun or referring expression has an unclear antecedent or could refer to multiple entities.

Coordination:
Coordination ambiguity occurs when words connected by conjunctions such as 'and' or 'or' allow multiple interpretations of grouping, scope, or relationships.

MissingCondition:
Missing Condition ambiguity occurs when a requirement omits a condition, constraint, or triggering circumstance needed to determine when or how the specified behavior should occur.


In [29]:

import re
import torch

# Model prediction function
def predict(prompt):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a software requirements ambiguity "
                "classifier. Return only 0 or 1."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[-1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # Extract binary prediction
    match = re.fullmatch(r"\s*([01])\s*\.?\s*", response)

    if match:
        return int(match.group(1))

    raise ValueError(
        f"Invalid model prediction: {response!r}"
    )

print("Prediction function created successfully!")


Prediction function created successfully!


In [30]:

# Zero-shot prompting

def build_zero_shot_prompt(requirement, ambiguity_type):

    definition = TYPE_DEFINITIONS[ambiguity_type]

    prompt = f"""You are evaluating a software requirement.

Ambiguity type: {ambiguity_type}

Definition:
{definition}

Classify the requirement for this specific ambiguity type.

Labels:
0 = Not ambiguous
1 = Ambiguous

Return only one digit: 0 or 1.

Requirement:
{requirement}

Answer:"""

    return prompt





In [31]:

# Fixed Few-shot: one example per class and type

fixed_examples = {}

for ambiguity_type in TYPE_DEFINITIONS:

    subset = development_df[
        development_df["Ambiguity_Type"] == ambiguity_type
    ]

    negative = subset[
        subset["Label"] == 0
    ].iloc[0]

    positive = subset[
        subset["Label"] == 1
    ].iloc[0]

    fixed_examples[ambiguity_type] = [
        {
            "Requirement": negative["Requirement"],
            "Label": 0
        },
        {
            "Requirement": positive["Requirement"],
            "Label": 1
        }
    ]

print("Fixed Few-shot examples prepared successfully!")

for ambiguity_type, examples in fixed_examples.items():
    print(f"\n{ambiguity_type}:")
    for example in examples:
        print("Label:", example["Label"])
        print("Requirement:", example["Requirement"])
        print("-" * 40)


Fixed Few-shot examples prepared successfully!

Anaphoric:
Label: 0
Requirement: Buyer can view status of book whether it is received or not.
----------------------------------------
Label: 1
Requirement: Supervisor can view and mark his attendance
----------------------------------------

Coordination:
Label: 0
Requirement: The application shall enable visitor to click back arrow which is offered at the top left corner of page or click back arrow provide by the device to go back to previous page.
----------------------------------------
Label: 1
Requirement: Registration requirements include name, username, email id, contact number, password and confirm password and the user username for the user.
----------------------------------------

MissingCondition:
Label: 0
Requirement: If system find Repairer, then customer should receive notification about sending the request successfully.
----------------------------------------
Label: 1
Requirement: The app shall display error message if t

In [32]:

# Fixed Few-shot Prompting

def build_fixed_few_shot_prompt(requirement, ambiguity_type):

    definition = TYPE_DEFINITIONS[ambiguity_type]
    examples = fixed_examples[ambiguity_type]

    examples_text = ""

    for example in examples:
        examples_text += (
            f"Requirement: {example['Requirement']}\n"
            f"Answer: {example['Label']}\n\n"
        )

    prompt = f"""You are evaluating a software requirement.

Ambiguity type: {ambiguity_type}

Definition:
{definition}

Classify the requirement for this specific ambiguity type.

Labels:
0 = Not ambiguous
1 = Ambiguous

Here are labeled examples:

{examples_text}
Return only one digit: 0 or 1.

Requirement:
{requirement}

Answer:"""

    return prompt



In [33]:

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load sentence embedding model
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(
    EMBEDDING_MODEL,
    device="cpu"
)

# Generate embeddings for Development Set
development_embeddings = embedder.encode(
    development_df["Requirement"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Validate embeddings
assert development_embeddings.shape[0] == len(development_df)

print("MiniLM loaded successfully!")
print("Development embeddings shape:", development_embeddings.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

MiniLM loaded successfully!
Development embeddings shape: (81, 384)


In [34]:

# Dynamic Few-shot: Semantic Example Selection

def select_dynamic_examples(requirement, ambiguity_type):

    # Encode the test requirement
    query_embedding = embedder.encode(
        [requirement],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Select Development Set examples of the same type
    candidate_indices = development_df.index[
        development_df["Ambiguity_Type"] == ambiguity_type
    ].to_numpy()

    candidate_embeddings = development_embeddings[
        candidate_indices
    ]

    # Calculate cosine similarity
    similarities = cosine_similarity(
        query_embedding,
        candidate_embeddings
    )[0]

    selected_examples = []

    # Select the most similar example from each class
    for label in [0, 1]:

        label_positions = [
            pos
            for pos, idx in enumerate(candidate_indices)
            if development_df.loc[idx, "Label"] == label
        ]

        if not label_positions:
            raise ValueError(
                f"No development example for "
                f"{ambiguity_type}, label {label}"
            )

        best_position = max(
            label_positions,
            key=lambda pos: similarities[pos]
        )

        best_index = candidate_indices[best_position]
        best_row = development_df.loc[best_index]

        selected_examples.append({
            "Requirement": best_row["Requirement"],
            "Label": int(best_row["Label"]),
            "Similarity": float(similarities[best_position])
        })

    return selected_examples



In [35]:

# Dynamic Few-shot Prompting

def build_dynamic_few_shot_prompt(requirement, ambiguity_type):

    definition = TYPE_DEFINITIONS[ambiguity_type]

    # Retrieve the most similar examples
    examples = select_dynamic_examples(
        requirement,
        ambiguity_type
    )

    examples_text = ""

    for example in examples:
        examples_text += (
            f"Requirement: {example['Requirement']}\n"
            f"Answer: {example['Label']}\n\n"
        )

    prompt = f"""You are evaluating a software requirement.

Ambiguity type: {ambiguity_type}

Definition:
{definition}

Classify the requirement for this specific ambiguity type.

Labels:
0 = Not ambiguous
1 = Ambiguous

Here are labeled examples:

{examples_text}
Return only one digit: 0 or 1.

Requirement:
{requirement}

Answer:"""

    return prompt





In [36]:

import pandas as pd
import time
from pathlib import Path

# Experiment configuration
METHODS = {
    "Zero-shot": build_zero_shot_prompt,
    "Fixed Few-shot": build_fixed_few_shot_prompt,
    "Dynamic Few-shot": build_dynamic_few_shot_prompt
}

OUTPUT_FILES = {
    "Zero-shot": "Qwen_ZeroShot_Results.xlsx",
    "Fixed Few-shot": "Qwen_FixedFewShot_Results.xlsx",
    "Dynamic Few-shot": "Qwen_DynamicFewShot_Results.xlsx"
}

def run_experiment(method_name, prompt_builder):

    output_file = OUTPUT_FILES[method_name]
    checkpoint_file = output_file.replace(
        ".xlsx", "_checkpoint.csv"
    )

    # Resume previously completed predictions if available
    if Path(checkpoint_file).exists():
        results = pd.read_csv(checkpoint_file)
        print(f"Resuming {method_name}: {len(results)} rows")
    else:
        results = pd.DataFrame()

    # Validate checkpoint before resuming
    if not results.empty:

        required_columns = [
            "Test_Index",
            "Requirement",
            "Ambiguity_Type",
            "Actual_Label",
            "Predicted_Label",
            "Error"
        ]

        if not set(required_columns).issubset(results.columns):
            raise ValueError(
                f"{method_name}: Invalid checkpoint columns"
            )

        if len(results) > len(test_df):
            raise ValueError(
                f"{method_name}: Checkpoint is too long"
            )

        expected = test_df.iloc[:len(results)]

        if (
            results["Test_Index"].tolist()
            != list(range(len(results)))
            or results["Requirement"].fillna("").astype(str).tolist()
            != expected["Requirement"].fillna("").astype(str).tolist()
            or results["Ambiguity_Type"].tolist()
            != expected["Ambiguity_Type"].tolist()
            or results["Actual_Label"].astype(int).tolist()
            != expected["Label"].astype(int).tolist()
        ):
            raise ValueError(
                f"{method_name}: Checkpoint does not match test data"
            )

        if (
            results["Predicted_Label"].isna().any()
            or not results["Predicted_Label"].isin([0, 1]).all()
            or not results["Error"].fillna("").eq("").all()
        ):
            raise ValueError(
                f"{method_name}: Checkpoint contains invalid predictions"
            )

    start_index = len(results)

    for i in range(start_index, len(test_df)):

        row = test_df.iloc[i]

        prompt = prompt_builder(
            row["Requirement"],
            row["Ambiguity_Type"]
        )

        try:
            prediction = predict(prompt)
            error_message = ""
        except Exception as e:
            prediction = None
            error_message = str(e)

        new_row = {
            "Test_Index": i,
            "Requirement": row["Requirement"],
            "Ambiguity_Type": row["Ambiguity_Type"],
            "Actual_Label": int(row["Label"]),
            "Predicted_Label": prediction,
            "Error": error_message
        }

        results = pd.concat(
            [results, pd.DataFrame([new_row])],
            ignore_index=True
        )

        # Save checkpoint every 25 requirements
        if (i + 1) % 25 == 0:
            results.to_csv(checkpoint_file, index=False)
            print(
                f"{method_name}: "
                f"{i + 1}/{len(test_df)} completed"
            )

    # Save final checkpoint and Excel results
    results.to_csv(checkpoint_file, index=False)
    results.to_excel(output_file, index=False)

    print(f"\n{method_name} completed!")
    print("Saved:", output_file)

    return results


# Run all three strategies
all_results = {}

for method_name, prompt_builder in METHODS.items():

    print("\n" + "=" * 50)
    print("Starting:", method_name)
    print("=" * 50)

    all_results[method_name] = run_experiment(
        method_name,
        prompt_builder
    )

print("\nAll three experiments completed!")



Starting: Zero-shot
Resuming Zero-shot: 324 rows

Zero-shot completed!
Saved: Qwen_ZeroShot_Results.xlsx

Starting: Fixed Few-shot
Resuming Fixed Few-shot: 324 rows

Fixed Few-shot completed!
Saved: Qwen_FixedFewShot_Results.xlsx

Starting: Dynamic Few-shot
Resuming Dynamic Few-shot: 324 rows

Dynamic Few-shot completed!
Saved: Qwen_DynamicFewShot_Results.xlsx

All three experiments completed!


In [37]:

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Verify all predictions before evaluation
for method_name, results in all_results.items():

    assert len(results) == len(test_df), (
        f"{method_name}: Incorrect number of predictions"
    )

    assert results["Test_Index"].tolist() == list(
        range(len(test_df))
    ), f"{method_name}: Test indices are incorrect"

    assert results["Predicted_Label"].notna().all(), (
        f"{method_name}: Missing predictions found"
    )

    assert results["Predicted_Label"].isin([0, 1]).all(), (
        f"{method_name}: Invalid predictions found"
    )

    assert results["Error"].fillna("").eq("").all(), (
        f"{method_name}: Prediction errors found"
    )

print("All predictions validated successfully!")

# Calculate evaluation metrics
evaluation_rows = []

for method_name, results in all_results.items():

    for ambiguity_type in [
        "Anaphoric",
        "Coordination",
        "MissingCondition",
        "Overall"
    ]:

        if ambiguity_type == "Overall":
            subset = results
        else:
            subset = results[
                results["Ambiguity_Type"] == ambiguity_type
            ]

        y_true = subset["Actual_Label"].astype(int)
        y_pred = subset["Predicted_Label"].astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1]
        ).ravel()

        evaluation_rows.append({
            "Method": method_name,
            "Ambiguity_Type": ambiguity_type,
            "N": len(subset),
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(
                y_true, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_true, y_pred, zero_division=0
            ),
            "F1": f1_score(
                y_true, y_pred, zero_division=0
            ),
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn
        })

comparison_df = pd.DataFrame(evaluation_rows)

# Save complete evaluation
comparison_df.to_excel(
    "Qwen_Final_Comparison.xlsx",
    index=False
)

print("\nFinal evaluation results:")
print(comparison_df.round(4).to_string(index=False))

print("\nSaved: Qwen_Final_Comparison.xlsx")
print("Evaluation completed successfully!")


All predictions validated successfully!

Final evaluation results:
          Method   Ambiguity_Type   N  Accuracy  Precision  Recall     F1  TP  TN  FP  FN
       Zero-shot        Anaphoric 127    0.5118     0.3333  0.1923 0.2439  10  55  20  42
       Zero-shot     Coordination 127    0.4567     0.4537  0.8305 0.5868  49   9  59  10
       Zero-shot MissingCondition  70    0.2571     0.0200  0.2500 0.0370   1  17  49   3
       Zero-shot          Overall 324    0.4352     0.3191  0.5217 0.3960  60  81 128  55
  Fixed Few-shot        Anaphoric 127    0.4803     0.3833  0.4423 0.4107  23  38  37  29
  Fixed Few-shot     Coordination 127    0.5276     0.4921  0.5254 0.5082  31  36  32  28
  Fixed Few-shot MissingCondition  70    0.1286     0.0615  1.0000 0.1159   4   5  61   0
  Fixed Few-shot          Overall 324    0.4228     0.3085  0.5043 0.3828  58  79 130  57
Dynamic Few-shot        Anaphoric 127    0.3937     0.3529  0.5769 0.4380  30  20  55  22
Dynamic Few-shot     Coordination

In [38]:

from google.colab import files
from pathlib import Path

result_files = [
    "Qwen_ZeroShot_Results.xlsx",
    "Qwen_FixedFewShot_Results.xlsx",
    "Qwen_DynamicFewShot_Results.xlsx",
    "Qwen_Final_Comparison.xlsx"
]

for filename in result_files:
    if not Path(filename).exists():
        raise FileNotFoundError(f"Missing file: {filename}")

print("All four result files are available.")

for filename in result_files:
    files.download(filename)


All four result files are available.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>